If you find this notebook helpful or elegant, please consider giving it an **Upvote**! 👍 It keeps me motivated to share more clean pipelines!

The score improved from 78.22 to 78.95 by adding the logistic model and random forest classifier!

# Titanic: Machine Learning from Disaster
### Comprehensive EDA & Soft-Voting Ensemble Pipeline

This notebook showcases a complete end-to-end predictive modeling workflow on the Titanic dataset, structured into:
1. Thorough Exploratory Data Analysis (EDA) & missing value visualizations.
2. Robust variable engineering (Deck Extraction, Family structures, and Ticket cleaning).
3. Data scaling and dimensional column alignment to safeguard test inference.
4. Stratified hyperparameter optimization across cross-validated architectures.
5. Soft-Voting Ensembling integrating tree architectures and geometric classifiers.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Suppress unneeded warnings
warnings.filterwarnings('ignore')

# Secure reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load data structures
df_train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
df_test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

## 1. Structural Exploration & Metadata Removal
### Analysis & Intent
`PassengerId` and `Name` are high-cardinality nominal values unique to individual rows. Passing them into mathematical estimators induces sparse splitting criteria or structural memorization, causing heavy overfitting. We drop these metadata columns immediately to ensure the baseline generalities focus on passenger dynamics.

In [ ]:
df_train = df_train.drop(columns=['PassengerId'])
df_test = df_test.drop(columns=['PassengerId'])

print("--- Training Data Overview ---")
print(df_train.head())
print("\n--- Training Descriptive Statistics ---")
print(df_train.describe())

print("\n--- Testing Structural Overview ---")
print(df_test.head())
print("\n--- Testing Descriptive Statistics ---")
print(df_test.describe(include='all').T)

print("\n--- Feature Cardinality Check ---")
for columns in df_train.columns:
    print(columns, ":", df_train[columns].nunique())

In [ ]:
import pandas as pd


# 敬称を抽出して新しい列 'Title' を作成
df_train['Title'] = df_train['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
df_test['Title'] = df_test['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# どんな敬称がどれくらいあるか確認
print(df_train['Title'].value_counts())
print(df_test['Title'].value_counts())

In [ ]:
title_mapping = {
    "Mr": "Mr", "Miss": "Miss", "Mrs": "Mrs", "Master": "Master",
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Lady": "Rare", "Countess": "Rare", "Capt": "Rare", "Col": "Rare",
    "Don": "Rare", "Dr": "Rare", "Major": "Rare", "Rev": "Rare", 
    "Sir": "Rare", "Jonkheer": "Rare", "Dona": "Rare"
}

df_train['Title'] = df_train['Title'].map(title_mapping)
df_train['Surname'] = df_train['Name'].apply(lambda x: x.split(',')[0].strip())
df_train['Title'] = df_train['Title'].astype('category')
df_train = df_train.drop(columns=['Name','Surname'])
print(df_train.head())

df_test['Title'] = df_test['Title'].map(title_mapping)
df_test['Surname'] = df_test['Name'].apply(lambda x: x.split(',')[0].strip())
df_test['Title'] = df_test['Title'].astype('category')
df_test = df_test.drop(columns=['Name','Surname'])
print(df_test.head())


## 2. Demographic Exploration & Categorical Encoding (Sex)
### Analysis & Domain Insights
Encoding the `Sex` element into binary vectors allows our distance matrices to map demographic dependencies. The resulting bar chart highlights a substantial rift: **the survival probability of female passengers overwhelmingly exceeds that of males**. This empirical finding confirms the traditional "women and children first" maritime protocol. We anticipate this feature to carry dominant weight across our models.

In [ ]:
# Label encoding for Sex
le = LabelEncoder()
df_train["Sex"] = le.fit_transform(df_train["Sex"])
df_test["Sex"] = le.transform(df_test["Sex"])

Sex_Survived = df_train.groupby('Sex')['Survived'].mean().reset_index()

plt.figure(figsize=(5, 4))
sns.barplot(data=Sex_Survived, x='Sex', y='Survived')
plt.title('Survival Ratio by Sex (0=Male, 1=Female)')
plt.show()

## 3. Structural Missing Values Identification
### Deep Dive Analysis
Using numeric reporting alongside null-value heatmaps allows us to locate patterns within missing records. `Cabin` contains a massive concentration of missing values (>77%), requiring structured grouping rather than simple row-dropping. `Age` presents mild sparsity (~20%), which requires a variance-preserving filling technique, while `Embarked` displays negligible missing rows.

In [ ]:
# Evaluate Missing Valuations in Training
missing_values = df_train.isnull().sum().sort_values(ascending=False)
missing_values_percentage = (missing_values / len(df_train)) * 100
missing = missing_values[missing_values_percentage > 0]
print("Missing Values (Train):\n", missing)

# Visualizing Missingness Patterns
plt.figure(figsize=(10, 6))
sns.heatmap(df_train.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap (Train Data)')
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(df_train['Age'], bins=30, kde=True)
plt.title('Age Distribution Pattern (Train Data)')
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(df_test.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap (Test Data)')
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(df_test['Age'], bins=30, kde=True)
plt.title('Age Distribution Pattern (Test Data)')
plt.show()

Age_Survived = df_train.groupby('Age')['Survived'].mean().reset_index()
plt.figure(figsize=(12, 5))
sns.barplot(data=Age_Survived, x='Age', y='Survived')
plt.title('Raw Survival Ratio across Age Groups')
plt.xticks(rotation=90)
plt.locator_params(axis='x', nbins=30)
plt.show()

## 4. Variance-Preserving Imputation (Age)
### Methodological Justification
Replacing missing continuous values with standard parameters (like mean or median) spikes the kurtosis artificially and flattens variances, altering the natural data shape. To protect the original age distribution, we draw random observations directly from the valid subset of training observations, ensuring our models process consistent statistical shapes.

In [ ]:
import pandas as pd

df_train['Title'] = df_train['Title'].astype(str)
df_test['Title'] = df_test['Title'].astype(str)

df_train['Age_is_missing'] = df_train['Age'].isnull().astype(int)
df_test['Age_is_missing'] = df_test['Age'].isnull().astype(int)


title_medians_dict = df_train.groupby('Title')['Age'].median().to_dict()

overall_median = df_train['Age'].median()

train_imputed_ages = df_train['Title'].map(title_medians_dict).fillna(overall_median)
df_train['Age'] = df_train['Age'].fillna(train_imputed_ages)

test_imputed_ages = df_test['Title'].map(title_medians_dict).fillna(overall_median)
df_test['Age'] = df_test['Age'].fillna(test_imputed_ages)


## 5. Port of Embarkation Processing
### Analysis & Environmental Mapping
The distribution plot shows that Port "S" (Southampton) represents the largest boarding location. Assessing port-grouped survival rates reveals that passengers who boarded at Port "C" (Cherbourg) experienced higher survival rates. This variation suggests a strong correlation with premium cabin allocations. We apply one-hot encoding with `drop_first` mechanics to neutralize multicollinearity risks.

In [ ]:
emb_mode = df_train["Embarked"].mode()[0]
df_train["Embarked"] = df_train["Embarked"].fillna(emb_mode)
df_test["Embarked"] = df_test["Embarked"].fillna(emb_mode)

plt.figure(figsize=(5, 4))
sns.countplot(x='Embarked', data=df_train)
plt.title('Passenger Concentration across Embarkation Ports')
plt.show()

embark_perc = df_train[["Embarked", "Survived"]].groupby(['Embarked'], as_index=False).mean()
plt.figure(figsize=(5, 4))
sns.barplot(x='Embarked', y='Survived', data=embark_perc, order=['S', 'C', 'Q'])
plt.title('Survival Ratios grouped by Embarkation Port')
plt.show() 

# Generate Dummy Dimensions and drop reference class 'S'
train_Embarked_dummies = pd.get_dummies(df_train['Embarked'], dtype=int).drop(['S'], axis=1)
test_Embarked_dummies = pd.get_dummies(df_test['Embarked'], dtype=int).drop(['S'], axis=1)

df_train = df_train.join(train_Embarked_dummies).drop(['Embarked'], axis=1)
df_test = df_test.join(test_Embarked_dummies).drop(['Embarked'], axis=1)

## 6. Multi-Feature Correlation Diagnosis
### Matrix Insights
Constructing a linear correlation matrix shows the deep dependencies within the dataset. We note strong inverse interactions between `Pclass` and `Fare`, indicating that higher class tiers command significantly higher pricing structures. Identifying these interactions helps ensure stable parameter weights across our downstream algorithms.

In [ ]:
numerical_df = df_train.select_dtypes(include=['number'])
corr_matrix = numerical_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Linear Correlation Topography')
plt.show()

## 7. Granular Estimation & Class-Stratified Imputation (Fare)
### Analysis & Contextual Imputation
Evaluating the distribution of `Fare` values shows a steep right-skewed pattern, indicating premium outliers. Boxplots and median comparisons show that fare valuations change dramatically based on `Pclass` and `Embarked` connections. Because the single missing fare entry resides in `Pclass 3`, we use the median value of `Pclass 3` from the training pool to handle the missing row accurately.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df_test['Fare'], kde=True, bins=50)
plt.title('Raw Test Fare Density Structure')
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x='Pclass', y='Fare', data=df_test, showfliers=False)
plt.title('Fare Distribution Spans across Passenger Classes')
plt.show()

df_plot = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
df_plot["Embarked"] = df_plot["Embarked"].fillna(df_plot["Embarked"].mode()[0])

plt.figure(figsize=(8, 5))
sns.barplot(x='Pclass', y='Fare', hue='Embarked', data=df_plot, estimator=lambda x: np.nanmedian(x)) 
plt.title('Median Fare Layout by Class and Embarkation Linkages')
plt.ylabel('Median Fare Column Values')
plt.show()

# Isolate Target Missing Profile
print("Target Fare Record Missing Profile:")
print(df_test[df_test['Fare'].isnull()])

pclass3_fare_median = df_train[df_train['Pclass'] == 3]['Fare'].median()
df_test['Fare'] = df_test['Fare'].fillna(pclass3_fare_median)

## 8. Feature Engineering (Cabin Decks & Family Structuring)
### Engineering Hypotheses
- **Cabin Deck**: The initial letter in the `Cabin` string indicates structural deck allocations (Decks A-G). Lower decks are more vulnerable to flooding and have longer paths to lifeboats. Missing cabins are labeled as 'U' (Unknown), serving as an informative proxy for unassigned tiers.
- **Family Structures**: Combining `SibSp` and `Parch` into a single `Family` metric lets us evaluate social group dynamics, while the `Alone` flag captures the survival differences between solo passengers and family units.

In [ ]:
df_train['Cabin_Letter'] = df_train['Cabin'].str[0].fillna('U')
df_test['Cabin_Letter']  = df_test['Cabin'].str[0].fillna('U')

df_train = pd.get_dummies(df_train, columns=['Cabin_Letter'], drop_first=True, dtype=int).drop(columns=['Cabin'])
df_test  = pd.get_dummies(df_test,  columns=['Cabin_Letter'], drop_first=True, dtype=int).drop(columns=['Cabin'])

def make_family_features(df):
    df['Family'] = df['SibSp'] + df['Parch'] + 1
    df['Alone'] = 0
    df.loc[df['Family'] == 1, 'Alone'] = 1
    return df.drop(columns=['SibSp', 'Parch'])

train_processed = make_family_features(df_train)
test_processed = make_family_features(df_test)

## 9. Categorical Parsing (Ticket Prefixes & Passenger Classes)
### Analysis & Structural Formatting
- **Ticket**: Extracting alpha-numeric ticket prefixes removes individual noise while grouping common booking pathways and localized purchase sources together.
- **Pclass**: Although stored as numbers, passenger classes are distinct ordinal tiers. We convert `Pclass` into string objects before running `pd.get_dummies` to avoid imposing an artificial linear structure on distance-based models.

In [ ]:
def clean_ticket(ticket):
    ticket = ticket.replace('.', '').replace('/', '').strip()
    parts = ticket.split()
    return parts[0] if len(parts) > 1 else 'X'

train_processed['ticket_letter'] = train_processed['Ticket'].apply(clean_ticket)
test_processed['ticket_letter'] = test_processed['Ticket'].apply(clean_ticket)

train_processed = train_processed.drop(columns=['Ticket'])
test_processed = test_processed.drop(columns=['Ticket'])

# Enforce Categorical Casting onto Pclass
train_processed['Pclass'] = train_processed['Pclass'].astype(str)
test_processed['Pclass'] = test_processed['Pclass'].astype(str)

train_processed = pd.get_dummies(train_processed, columns=['ticket_letter', 'Pclass'], drop_first=True, dtype=int)
test_processed = pd.get_dummies(test_processed, columns=['ticket_letter', 'Pclass'], drop_first=True, dtype=int)

## 10. Alignment & Scaling Pipeline
### Pipeline Stabilization Analysis
String-parsing can create different column setups if specific ticket categories are missing in either the training or testing splits. We use an explicit structural alignment check via `X.align` to ensure identical data shapes across both sets. Afterward, standard scaling balances continuous measurements (`Age`, `Fare`) to protect distance-sensitive models from feature magnitude imbalances.

In [ ]:
y = train_processed['Survived']
X = train_processed.drop(columns=['Survived'])

if 'Survived' in test_processed.columns:
    test_processed = test_processed.drop(columns=['Survived'])

# CRITICAL SECURITY STEP: Align shapes between sets to resolve category discrepancies
X, test_processed = X.align(test_processed, join='left', axis=1, fill_value=0)

scaler = StandardScaler()
X[['Age', 'Fare']] = scaler.fit_transform(X[['Age', 'Fare']])
test_processed[['Age', 'Fare']] = scaler.transform(test_processed[['Age', 'Fare']])

# Split into localized sets for stable testing validation
X_train, X_test_val, y_train, y_test_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 11. Hyperparameter Optimization & Model Exploration
### Tuning Overview
We optimize four distinct architectures (XGBoost and LightGBM) using grid search under Stratified K-Fold cross-validation. This setup ensures stable survival proportions across every validation fold.

In [ ]:
from sklearn.preprocessing import LabelEncoder

X_train = X_train.copy()
X_test_val = X_test_val.copy()

if 'test_processed' in globals():
    test_processed = test_processed.copy()

for col in ['Title']:
    if col in X_train.columns:
        le = LabelEncoder()
        X_train.loc[:, col] = le.fit_transform(X_train[col].astype(str))        
        X_test_val.loc[:, col] = X_test_val[col].astype(str).map(
            lambda s: le.transform([s])[0] if s in le.classes_ else -1
        )
        
        if 'test_processed' in globals() and col in test_processed.columns:
            test_processed.loc[:, col] = test_processed[col].astype(str).map(
                lambda s: le.transform([s])[0] if s in le.classes_ else -1
            )
        
        X_train[col] = X_train[col].astype(int)
        X_test_val[col] = X_test_val[col].astype(int)
        
        if 'test_processed' in globals() and col in test_processed.columns:
            test_processed[col] = test_processed[col].astype(int)

# --- 1. XGBoost --- 
xgb = XGBClassifier(random_state=42, eval_metric='logloss')

param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2]
}
grid_search_xgb = GridSearchCV(estimator=xgb, param_grid=param_grid_xgb, cv=skf, scoring='accuracy', n_jobs=-1)
grid_search_xgb.fit(X_train, y_train)

print("\n--- XGBoost Optimization Matrix ---")
print("Best Parameters:", grid_search_xgb.best_params_)
print(f"Best CV Accuracy Score: {grid_search_xgb.best_score_:.4f}")

best_xgb_model = grid_search_xgb.best_estimator_
xgb_preds = best_xgb_model.predict(X_test_val)



In [ ]:
# --- 2. LightGBM ---
lgb = LGBMClassifier(random_state=42, objective='binary', verbosity=-1)

param_grid_lgb = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'num_leaves': [7, 15, 31],
    'learning_rate': [0.01, 0.05, 0.1]
}
grid_search_lgb = GridSearchCV(estimator=lgb, param_grid=param_grid_lgb, cv=skf, scoring='accuracy', n_jobs=-1)
grid_search_lgb.fit(X_train, y_train)

print("\n--- LightGBM Optimization Matrix ---")
print("Best Parameters:", grid_search_lgb.best_params_)
print(f"Best CV Accuracy Score: {grid_search_lgb.best_score_:.4f}")

best_lgb_model = grid_search_lgb.best_estimator_
lgb_preds = best_lgb_model.predict(X_test_val)


In [ ]:
from sklearn.ensemble import  RandomForestClassifier
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 7, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}
grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf, cv=skf, scoring='accuracy', n_jobs=-1, verbose=0)
grid_search_rf.fit(X_train, y_train)

print(f"   Best CV Score: {grid_search_rf.best_score_:.4f}")
best_rf_model = grid_search_rf.best_estimator_
rf_preds = best_rf_model.predict(X_test_val)


In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(random_state=42, max_iter=1000)

param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear']
}
grid_search_lr = GridSearchCV(estimator=lr, param_grid=param_grid_lr, cv=skf, scoring='accuracy', verbose=0)
grid_search_lr.fit(X_train, y_train)

print(f"   Best CV Score: {grid_search_lr.best_score_:.4f}")
best_lr_model = grid_search_lr.best_estimator_
lr_acc = accuracy_score(y_test_val, best_lr_model.predict(X_test_val))


## 12. Ensemble Consensus (Soft Voting) & Localized Assessment
### Ensemble Strategy Analysis
We combine our optimized models into a soft-voting ensemble. By averaging predicted class probabilities instead of relying on hard voting choices, this approach effectively balances individual model errors. This strategy reduces variance and enhances generalization against unseen test distributions.

In [ ]:
best_estimators = [
    ('xgb', grid_search_xgb.best_estimator_),
    ('lgb', grid_search_lgb.best_estimator_),
    ('rf', best_rf_model),
    ('lr', best_lr_model)
]

ensemble_model = VotingClassifier(estimators=best_estimators, voting='soft')
ensemble_model.fit(X_train, y_train)

final_preds_ensemble = ensemble_model.predict(X_test_val)

print("--- Final Localized Validation Metrics (Ensemble) ---")
print(f"Accuracy Metric: {accuracy_score(y_test_val, final_preds_ensemble):.4f}")
print("\nClassification Report Summary:")
print(classification_report(y_test_val, final_preds_ensemble))

print("\nConfusion Matrix Structure:")
print(confusion_matrix(y_test_val, final_preds_ensemble))

## 13. Test Inferences & Kaggle Output Production
### Final Step
The trained ensemble handles inferences directly on the aligned test split. We combine these output maps with the original tracking indexes to generate a clean, properly formatted `submission.csv` ready for Kaggle submission.

In [ ]:
for col in ['Title', 'Surname']:
    if col in test_processed.columns:
        test_processed[col] = test_processed[col].astype(int)

final_test_predictions = ensemble_model.predict(test_processed[X_train.columns])

df_original_test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

submission_output = pd.DataFrame({
    'PassengerId': df_original_test['PassengerId'],
    'Survived': final_test_predictions
})

submission_output.to_csv('submission.csv', index=False)


Thank u for visiting my Notebook!